In [1]:
import os
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Report summary

In [2]:
def str2int(s):
    return int(s.replace(",", ""))
    
dat = pd.read_excel("../../data/NASCseq.xlsx")

In [3]:
# trimming
vs1 = [] # total reads
vs2 = [] # trimmed reads
for run, cell in dat[["Run", "Cell"]].values:
    total_reads = 0
    trimmed_reads = 0
    path = "../../results/1_nascseq/1_fastqs/cutadapt/%s/%s.log" % (run, cell)
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if "Total reads processed:" in line or "Total read pairs processed:" in line:
                    total_reads = str2int(line.strip("\n").split()[-1])
                if "Reads written (passing filters):" in line or "Pairs written (passing filters):" in line:
                    trimmed_reads = str2int(line.strip("\n").split()[-2])
    vs1.append(total_reads)
    vs2.append(trimmed_reads)
dat["Total.Reads"] = vs1
dat["Trimmed.Reads"] = vs2
dat["Trimmed.Reads.Ratio"] = dat["Trimmed.Reads"] / dat["Total.Reads"]

In [5]:
# mapping
vs1 = [] # reads
vs2 = [] # uniq mapped
for run, cell in dat[["Run", "Cell"]].values:
    reads = 0
    uniq_mapped = 0
    path = "../../results/1_nascseq/2_bams/star/%s/%s.Log.final.out" % (run, cell)
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if "Number of input reads" in line:
                    reads = int(line.strip().split()[-1])
                if "Uniquely mapped reads number" in line:
                    uniq_mapped = int(line.strip().split()[-1])
    vs1.append(reads)
    vs2.append(uniq_mapped)
dat["RiboRNA.Ratio"] = 1 - np.array(vs1) / dat["Trimmed.Reads"]
dat["Clean.Reads"] = vs1
dat["UniqMapped.Reads"] = vs2
dat["UniqMapped.Ratio"] = dat["UniqMapped.Reads"] / dat["Clean.Reads"]

In [6]:
# filtering

vs = []
for run, cell in dat[["Run", "Cell"]].values:
    reads = 0
    path = "../../results/1_nascseq/2_bams/filtered/%s/%s.flagstat" % (run, cell)
    if os.path.exists(path):
        for line in open(path):
            if "in total" in line:
                reads = int(line.split()[0])
                reads = int(reads / 2)
    vs.append(reads)
dat["Filtered.Reads"] = vs

In [7]:
# mark strand

vs1, vs2, vs3, = [], [], []
for run, cell in dat[["Run", "Cell"]].values:
    # mark strand
    uniq, pos, neg = np.nan, np.nan, np.nan
    path = "../../results/1_nascseq/2_bams/markstrand/%s/%s.tsv" % (run, cell)
    if os.path.exists(path):
        d = pd.read_csv(path, sep="\t", index_col=0)
        vs = d.values[0,:]
        uniq = sum(vs)
        pos, neg = vs[0], vs[1]
    vs1.append(uniq)
    vs2.append(pos)
    vs3.append(neg)
dat["Uniq.Reads"] = vs1
dat["Uniq.Ratio"] = dat["Uniq.Reads"] / dat["Filtered.Reads"]
dat["Forward"], dat["Reverse"] = vs2, vs3
dat["Stranded.Reads"] = dat["Forward"] + dat["Reverse"]
dat["Stranded.Ratio"] = dat["Stranded.Reads"] / dat["Uniq.Reads"]

In [8]:
# new reads
vs = []
for run, cell in dat[["Run", "Cell"]].values:
    path = "../../results/1_nascseq/2_bams/marknew/%s/%s.tsv" % (run, cell)
    d = pd.read_csv(path, sep="\t")
    vs.append(d["New"].values[0])
dat["New.Reads"] = vs
dat["New.Reads.Ratio"] = dat["New.Reads"] / dat["Stranded.Reads"]

In [9]:
# mismatch ratio
mtypes = []
for b1 in "ACGT":
    for b2 in "ACGT":
        if b1 != b2:
            mtypes.append("%s%s" % (b1, b2))
rows = []
for run, cell in dat[["Run", "Cell"]].values:
    path = "../../results/1_nascseq/3_mismatch_ratio/marknew/%s/%s.tsv" % (run, cell)
    d = pd.read_csv(path, sep="\t", index_col=0)
    rows.append([d.loc[mt]["Ratio"] for mt in mtypes])
tmp = pd.DataFrame(rows, columns=["%s.Ratio" % mt for mt in mtypes])
for c in tmp.columns:
    dat[c] = tmp[c]

In [11]:
# genes
array = []
for run, cell in dat[["Run", "Cell"]].values:
    path = "../../results/1_nascseq/5_stats/gene_number/marknew/%s/%s.tsv" % (run, cell)
    array.append(pd.read_csv(path, sep="\t"))
tmp = pd.concat(array, axis=0, ignore_index=True)
for c in tmp.columns:
   dat["Genes.%s" % c] = tmp[c]

In [13]:
# Pe and Pc
array = []
for run, cell in dat[["Run", "Cell"]].values:
    path = "../../results/1_nascseq/5_stats/snr/%s/%s.csv" % (run, cell)
    array.append(pd.read_csv(path, sep=","))
tmp = pd.concat(array, axis=0, ignore_index=True)
tmp = tmp[["Pe", "Pc", "SNR"]]
for c in tmp.columns:
   dat[c] = tmp[c]

In [14]:
m = dat.copy()
m.index = m["Cell"]
m.drop(columns=["Cell"], inplace=True)
m.to_csv("results/NASCseq_Summary.csv")